# THEMIS v2 — LoRA Fine-tuning on Kaggle T4

**Model:** Mistral 7B Instruct v0.3 (4-bit NF4)
**Method:** LoRA rank=16, alpha=32, all attention modules
**Dataset:** 20,909 Indian law Q&A pairs
**Target:** >70% citation accuracy on BNS queries

---

### Before you start

1. Go to **Settings** (right panel) → **Add secret**
2. Add a secret named `HF_TOKEN` with your HuggingFace token (get one at https://huggingface.co/settings/tokens)
3. Make sure **GPU** is enabled: Settings → Accelerator → **GPU T4 x2**
4. Upload `dataset.json` to `/kaggle/input/` or paste the Kaggle dataset URL below

## Cell 1 — Install Dependencies

Unsloth provides 2x faster LoRA training with 60% less VRAM. We also install `trl` (SFTTrainer), `datasets`, and `peft` for the training pipeline.

**First run takes ~2 minutes.** Subsequent runs skip if already installed.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps trl transformers datasets peft accelerate bitsandbytes

## Cell 2 — Verify GPU

Confirm the T4 GPU is available. Training will be extremely slow on CPU — **do not proceed without a GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")
    print("Ready to train.")
else:
    raise RuntimeError("No GPU found! Go to Settings → Accelerator → GPU T4 x2")

## Cell 3 — Load Base Model (4-bit quantized)

We load Mistral 7B Instruct v0.3 in **4-bit NF4 quantization** using BitsAndBytes. This reduces VRAM usage from ~14GB (fp16) to ~4GB, leaving room for training on a T4.

Unsloth's `FastLanguageModel` handles quantization and memory optimization automatically.

**First run downloads ~4GB from HuggingFace. Takes ~2 minutes on Kaggle.**

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (fp16 on T4)
    load_in_4bit=True,    # NF4 quantization
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

## Cell 4 — Add LoRA Adapters

LoRA (Low-Rank Adaptation) adds small trainable matrices to each attention layer. Instead of updating all 7B parameters, we only train ~0.1% of them.

**v2 LoRA config:**
- **Rank (r=16):** Controls adapter capacity. v1 used r=8 — doubling captures more legal knowledge.
- **Alpha (32):** Scaling factor. Alpha/rank = 2x scaling.
- **Target modules:** All 4 attention projections (q, k, v, o). v1 only targeted q and v.
- **Dropout (0.05):** Prevents overfitting on the 20k dataset.

**Trainable parameters:** ~8.4M out of 7B (~0.12%)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                  # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32,                         # Scaling factor
    lora_dropout=0.05,                     # Regularization
    bias="none",
    use_gradient_checkpointing=True,       # Saves VRAM
    random_state=42,
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## Cell 5 — Load Dataset

The dataset is in **Alpaca instruction format**:
```json
{
  "instruction": "What is the punishment for theft under BNS?",
  "input": "",
  "output": "Under Section 303 of the Bharatiya Nyaya Sanhita..."
}
```

Each sample is formatted as:
```
### Instruction:
{instruction}

### Response:
{output}
```

**Upload `dataset.json` to Kaggle Datasets first, then paste the URL below.**

Or if you uploaded it to `/kaggle/input/themis-dataset/`, update the path accordingly.

In [ ]:
import json
from datasets import Dataset

# ============================================================
# OPTION A: Upload dataset.json to Kaggle Datasets
#   1. Go to kaggle.com → Datasets → New Dataset
#   2. Upload dataset.json
#   3. Copy the dataset URL and paste below
# ============================================================
DATASET_PATH = "/kaggle/input/themis-dataset/dataset.json"  # Update this path

# ============================================================
# OPTION B: Upload dataset.json directly to this notebook
#   Use the + Data button → Upload → select dataset.json
#   Then set DATASET_PATH to the uploaded file path
# ============================================================
# DATASET_PATH = "/kaggle/input/dataset.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} training samples")
print(f"Sample instruction: {raw_data[0]['instruction'][:80]}...")
print(f"Sample output length: {len(raw_data[0]['output'])} chars")

dataset = Dataset.from_list(raw_data)

## Cell 6 — Data Formatting Function

This function converts each JSON sample into the Alpaca prompt format that Mistral was instruction-tuned on.

The model learns to generate the `### Response:` section given an `### Instruction:`. This is the standard SFT (Supervised Fine-Tuning) approach.

In [ ]:
def format_instruction(sample):
    """Format a single sample for SFT training."""
    text = f"### Instruction:\n{sample['instruction']}\n\n"
    if sample.get("input"):
        text += f"### Input:\n{sample['input']}\n\n"
    text += f"### Response:\n{sample['output']}"
    return text

# Preview formatted output
print("=" * 60)
print("FORMATTED TRAINING SAMPLE")
print("=" * 60)
print(format_instruction(raw_data[0]))
print("=" * 60)
print(f"\nTokens (approx): {len(tokenizer.encode(format_instruction(raw_data[0])))}")

## Cell 7 — Configure Training

Training hyperparameters for v2:

| Parameter | v1 | v2 | Why changed |
|-----------|-----|-----|-------------|
| Epochs | 3 | 3 | Same — more data means fewer epochs needed |
| Batch size | 1 | 1 | T4 VRAM limit |
| Grad accum | 8 | 8 | Effective batch = 8 samples per step |
| Learning rate | 2e-4 | 2e-4 | Proven stable for LoRA |
| Max seq len | 512 | **1024** | Longer statutory text |
| Warmup | 3% | 3% | Stable warmup |
| Optimizer | adamw_8bit | adamw_8bit | VRAM efficient |

**Estimated training time:** ~2-3 hours on T4

**Estimated steps:** 20,909 samples ÷ 8 effective batch = ~2,614 steps × 3 epochs = ~7,842 steps total

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    formatting_func=format_instruction,
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,                    # T4 uses fp16, not bf16
        logging_steps=10,             # Log every 10 steps
        output_dir="themis-v2-outputs",
        optim="adamw_8bit",
        seed=42,
        save_strategy="steps",
        save_steps=500,               # Checkpoint every 500 steps
        save_total_limit=2,           # Keep only last 2 checkpoints
    ),
)

print("Trainer configured.")
print(f"Epochs: 3")
print(f"Effective batch size: {1 * 8} = 8")
print(f"Total steps: ~{len(dataset) * 3 // 8:,}")

## Cell 8 — Train

This is the main training loop. You'll see loss decreasing over time.

**What to watch for:**
- **Loss should decrease** from ~2.0 to ~0.3-0.5 over training
- **If loss spikes** to >5.0, the learning rate may be too high (unlikely with these settings)
- **If loss plateaus** early, the model may be overfitting (check eval loss if available)

**Runtime:** ~2-3 hours on T4. **Do not interrupt.**

The training will print progress every 10 steps. You can monitor loss in the output.

In [ ]:
import time

start = time.time()
print("Starting THEMIS v2 training...")
print(f"Dataset: {len(dataset)} samples")
print(f"Epochs: 3")
print(f"Estimated time: 2-3 hours")
print("-" * 60)

trainer_stats = trainer.train()

elapsed = time.time() - start
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)

print("-" * 60)
print(f"Training complete in {hours}h {minutes}m")
print(f"Final training loss: {trainer_stats.training_loss:.4f}")
print(f"Total steps: {trainer_stats.global_step}")

## Cell 9 — Save LoRA Adapter Locally

Save the trained LoRA adapter (small ~30MB file) to the Kaggle working directory.

This adapter contains only the trained weight deltas — you still need the base model to run inference.

In [ ]:
from pathlib import Path

ADAPTER_DIR = "./themis-lora-v2"
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check adapter size
adapter_files = list(Path(ADAPTER_DIR).glob("*"))
total_size = sum(f.stat().st_size for f in adapter_files if f.is_file())

print(f"Adapter saved to: {ADAPTER_DIR}")
print(f"Files: {len(adapter_files)}")
print(f"Total size: {total_size / 1e6:.1f} MB")
print(f"\nFiles saved:")
for f in sorted(adapter_files):
    if f.is_file():
        print(f"  {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

## Cell 10 — Push to HuggingFace Hub

Upload the adapter to HuggingFace Hub so it can be loaded from anywhere.

This uses the `HF_TOKEN` secret you configured at the start.

**The adapter will be pushed to:** `Daniel2503/themis-mistral-7b-lora`

This **overwrites** the v1 adapter. If you want to keep v1, create a new repo first (e.g., `Daniel2503/themis-mistral-7b-lora-v2`).

In [ ]:
import os
from huggingface_hub import HfApi, login

# ============================================================
# OPTION A: Use the HF_TOKEN secret (recommended)
# ============================================================
HF_TOKEN = os.environ.get("HF_TOKEN")

# ============================================================
# OPTION B: Paste token directly (NOT recommended for sharing)
# ============================================================
# HF_TOKEN = "hf_your_token_here"

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found! Add it as a Kaggle Secret:\n"
        "Settings → Add Secret → Name: HF_TOKEN, Value: your token"
    )

login(token=HF_TOKEN)
print("Logged into HuggingFace Hub.")

# Push adapter
REPO_NAME = "Daniel2503/themis-mistral-7b-lora"  # Change if you want a new repo

api = HfApi()
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO_NAME,
    repo_type="model",
)

print(f"\nAdapter uploaded to: https://huggingface.co/{REPO_NAME}")

## Cell 11 — Quick Test

Load the trained model and test it with a sample legal question to verify it works.

In [ ]:
# Load the trained model for testing
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading base model + trained adapter for testing...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

test_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

test_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
test_model = PeftModel.from_pretrained(test_model, ADAPTER_DIR)
test_model.eval()

print("Model loaded. Running test query...\n")

# Test query
test_question = "What is the punishment for theft under the Bharatiya Nyaya Sanhita?"
prompt = f"### Instruction:\n{test_question}\n\n### Response:\n"

inputs = test_tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
    )

response = test_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Q: {test_question}")
print(f"\nA: {response}")

## Cell 12 — Download Adapter

Download the trained adapter to your local machine for offline use.

In [ ]:
import shutil

# Zip the adapter for easy download
shutil.make_archive("themis-lora-v2", "zip", ".", ADAPTER_DIR)

print("Adapter zipped as themis-lora-v2.zip")
print("\nTo download: Right-click the file in Kaggle's output panel → Download")
print(f"\nAlso available at: https://huggingface.co/{REPO_NAME}")